# 04 · 64×64 DCGAN 학습
원본: Music_gen/music_gen_sil_geon_64.ipynb


DCGAN 모델 및 학습 루프는 TensorFlow Authors의 DCGAN 튜토리얼을 바탕으로
음악 이미지에 맞게 수정한 프로젝트 코드입니다. Copyright 2019 The TensorFlow Authors.
해당 기반 코드에는 Apache License 2.0이 적용됩니다. 저장소의 THIRD_PARTY_NOTICES.md와
LICENSES/Apache-2.0.txt를 참고하세요. 공개용 정리 과정에서 경로·셀 순서·설명을 수정했습니다.


입력은 03에서 만든 배열입니다. 학습에는 GPU 런타임을 권장합니다. 모델 구조·손실·배치 크기는 원본을 보존했습니다. 장시간 학습은 마지막 셀의 RUN_TRAINING을 True로 바꿔 시작합니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
from pathlib import Path
import os
import time
print("TensorFlow", tf.__version__)

In [ ]:
BASE = Path('/content/project')
DATASET = BASE / 'data' / 'data_set_64.npy'
checkpoint_dir = str(BASE / 'checkpoints')
IMAGE_DIR = BASE / 'outputs' / 'training'
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
train_images = np.load(DATASET, allow_pickle=False)
if train_images.ndim != 3 or train_images.shape[1:] != (64, 64) or len(train_images) == 0:
    raise ValueError('학습 배열은 비어 있지 않은 (N, 64, 64)여야 합니다.')
if not np.isin(train_images, [0, 1]).all():
    raise ValueError('정규화 전의 이진 피아노롤이 필요합니다.')

In [ ]:
train_images = train_images.reshape(train_images.shape[0], 64, 64, 1).astype('float32')
train_images = (train_images - 0.5) / 0.5 #정규화

In [ ]:
BUFFER_SIZE = train_images.shape[0] #train_images 개수랑 같았음
BATCH_SIZE = 256

In [ ]:
# 데이터 배치를 만들고 섞습니다.
train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

In [ ]:
#생성모델 만들기

def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))
    assert model.output_shape == (None, 8, 8, 256) # 주목: 배치사이즈로 None이 주어집니다.

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    assert model.output_shape == (None, 8, 8, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    # assert model.output_shape == (None, 16, 16, 64)
    # model.add(layers.BatchNormalization())
    # model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 16, 16, 64)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 32, 32, 32)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())


    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 64, 64, 1)

    return model

In [ ]:
generator = make_generator_model()

In [ ]:
#감별자 모델

# def make_discriminator_model():
#     model = tf.keras.Sequential()
#     model.add(layers.Conv2D(16, (5, 5), strides=(2, 2), padding='same',
#                                      input_shape=[128, 128, 1]))
#     model.add(layers.LeakyReLU())
#     model.add(layers.Dropout(0.3))

#     model.add(layers.Conv2D(32, (5, 5), strides=(2, 2), padding='same'))
#     model.add(layers.LeakyReLU())
#     model.add(layers.Dropout(0.3))

#     model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'))
#     model.add(layers.LeakyReLU())
#     model.add(layers.Dropout(0.3))

#     model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
#     model.add(layers.LeakyReLU())
#     # model.add(layers.Activation('sigmoid'))
#     model.add(layers.Dropout(0.3))

#     model.add(layers.Flatten())
#     model.add(layers.Dense(1))

#     return model

def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     input_shape=[64, 64, 1]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

In [ ]:
discriminator = make_discriminator_model()

In [ ]:
# 이 메서드는 크로스 엔트로피 손실함수 (cross entropy loss)를 계산하기 위해 헬퍼 (helper) 함수를 반환합니다.
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
cross_entropy

In [ ]:
#손실함수

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
#옵티마이저

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

In [ ]:
#체크포인트

# checkpoint_dir is configured above
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
#훈련파라미터

EPOCHS = 20000
noise_dim = 100
num_examples_to_generate = 16

# 이 시드를 시간이 지나도 재활용하겠습니다. 
# (GIF 애니메이션에서 진전 내용을 시각화하는데 쉽기 때문입니다.) 

seed = tf.random.normal([num_examples_to_generate, noise_dim])
# seed = tf.random.uniform([num_examples_to_generate, noise_dim], minval=-1, maxval=2, dtype=tf.int32)
# seed = tf.cast(seed, tf.float32)

In [ ]:
# `tf.function`이 어떻게 사용되는지 주목해 주세요.
# 이 데코레이터는 함수를 "컴파일"합니다.
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])
    # noise = tf.random.uniform([BATCH_SIZE, noise_dim], minval=-1, maxval=1, dtype=tf.int32)
    # noise = tf.cast(noise, tf.float32)

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
      generated_images = generator(noise, training=True)

      real_output = discriminator(images, training=True)
      fake_output = discriminator(generated_images, training=True)

      gen_loss = generator_loss(fake_output)
      disc_loss = discriminator_loss(real_output, fake_output)

    tf.print(disc_loss)
    tf.print(gen_loss)
    

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

In [ ]:
def generate_and_save_images(model, epoch, test_input):
  # `training`이 False로 맞춰진 것을 주목하세요.
  # 이렇게 하면 (배치정규화를 포함하여) 모든 층들이 추론 모드(inference mode, 학습된 모델로 결과물을 내는 것)로 실행됩니다. 
  predictions = (model(test_input, training=False) *0.5) +0.5

  fig = plt.figure(figsize=(8,8))

  for i in range(predictions.shape[0]):
      plt.subplot(4, 4, i+1)
      plt.imshow(predictions[i, :, :, 0], cmap='gray', origin='lower')
      plt.axis('off')

  if epoch > 10000 and epoch % 50 == 0:
    plt.savefig(str(IMAGE_DIR / 'image_at_epoch_{:05d}.png'.format(epoch)))
  plt.show()
  plt.close(fig)

In [ ]:
def train(dataset, epochs):
  for epoch in range(epochs):
    start = time.time()

    for image_batch in dataset:
      train_step(image_batch)

    # GIF를 위한 이미지를 바로 생성합니다.
    display.clear_output(wait=True)
    generate_and_save_images(generator,
                             epoch + 1,
                             seed)

    # 10000이상 에포크에서 50 에포크가 지날 때마다 모델을 저장합니다.
    if (epoch+1) > 10000 and (epoch + 1) % 50 == 0:
      checkpoint.save(file_prefix = checkpoint_prefix)
    
    # print (' 에포크 {} 에서 걸린 시간은 {} 초 입니다'.format(epoch +1, time.time()-start))
    print ('Time for epoch {} is {} sec'.format(epoch + 1, time.time()-start))
    # print(list_disc_loss, len(list_disc_loss))
    # print(list_gen_loss, len(list_gen_loss))

  # 마지막 에포크가 끝난 후 생성합니다.
  display.clear_output(wait=True)
  generate_and_save_images(generator,
                           epochs,
                           seed)

원본은 10,000 epoch 이후 50 epoch마다 체크포인트를 저장합니다. 공개본은 조기 종료한 학습도 남길 수 있도록 정상 종료 후 한 번 더 저장합니다. 생성 이미지 파일명의 epoch+10000 오프셋은 실제 epoch로 수정했습니다. EPOCHS=20000은 코드 설정값이며 최종 발매 모델의 학습 횟수 확정값은 아닙니다.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    train(train_dataset, EPOCHS)
    checkpoint.save(file_prefix=checkpoint_prefix)
else:
    print('설정과 경로 확인 후 RUN_TRAINING = True로 학습을 시작하세요.')